### Import libraries

In [1]:
import pandas as pd
import glob
from nltk import word_tokenize, pos_tag, sent_tokenize

### Read the data

In [19]:
df = pd.read_csv('../data/civility/convo_data_with_moderation_scores.csv')

# Rename convo_turn for clarity
df['convo_turn'] = df['convo_turn'].apply(lambda x: "user_response_1" if x=="initial_opinion" else x)

# Convert string representation of dict back to dict
df['emotion_scores'] = df['emotion_scores'].apply(lambda x: eval(x))

# Agree vs Disagree
df['agree_disagree'] = "Agree"
df.loc[df['treatment'].apply(lambda x: "Disagree" in x), 'agree_disagree'] = "Disagree"

# Ingroup vs Outgroup
df['ingroup_outgroup'] = "Ingroup"
df.loc[df['treatment'].apply(lambda x: "Outgroup" in x), 'ingroup_outgroup'] = "Outgroup"

### go emotions BERT model outputs

In [20]:
list_of_emotions = [*df['emotion_scores'].iloc[0].keys()]

In [21]:
# take column of dicts and make each key a separate column within the dataframe
emotion_df = pd.json_normalize(df['emotion_scores'])
df = pd.concat([df, emotion_df], axis=1)

In [22]:
df['convo_turn'] = df['convo_turn'].apply(lambda x: {'initial_opinion':'user_response_1'}.get(x, x))

In [23]:
pivot_df = df[df['convo_turn'].apply(lambda x: 'user' in x.lower())].pivot_table(index='treatment', values=list_of_emotions, aggfunc='mean')

for emo in list_of_emotions:
    print(f"Emotion: {emo}")
    print(" \n".join([f"{i[0]}: {i[1]}" for i in pivot_df[emo].items()]))
    print()

Emotion: neutral
Ingroup Agree: 0.22486399783298183 
Ingroup Disagree: 0.29680706973032145 
Outgroup Agree: 0.2711841588392556 
Outgroup Disagree: 0.3159952550737953

Emotion: approval
Ingroup Agree: 0.37377150512398466 
Ingroup Disagree: 0.28969686093086827 
Outgroup Agree: 0.3097653771026503 
Outgroup Disagree: 0.24981946309293973

Emotion: realization
Ingroup Agree: 0.030217378412942975 
Ingroup Disagree: 0.035147433984420565 
Outgroup Agree: 0.029541275728174103 
Outgroup Disagree: 0.03545683082032115

Emotion: disappointment
Ingroup Agree: 0.05747743113478702 
Ingroup Disagree: 0.055113685419061864 
Outgroup Agree: 0.04721524924254608 
Outgroup Disagree: 0.06199614855482119

Emotion: disapproval
Ingroup Agree: 0.0566638478016536 
Ingroup Disagree: 0.13129565253241424 
Outgroup Agree: 0.0973441472644421 
Outgroup Disagree: 0.1392985958750404

Emotion: optimism
Ingroup Agree: 0.05917445338783725 
Ingroup Disagree: 0.04516023943779708 
Outgroup Agree: 0.05518671485802071 
Outgroup Di

### Question asking

In [24]:
df[df["convo_turn"].apply(lambda x: "user" in x.lower())].pivot_table(index="treatment", values='question_flag')

,question_flag
treatment,
Ingroup Agree,0.101633
Ingroup Disagree,0.089938
Outgroup Agree,0.114661
Outgroup Disagree,0.128834


In [25]:
df[df["convo_turn"].apply(lambda x: "llm" in x.lower())].pivot_table(index="treatment", values='question_flag')

,question_flag
treatment,
Ingroup Agree,0.066061
Ingroup Disagree,0.216460
Outgroup Agree,0.062321
Outgroup Disagree,0.195518


### Gratitude usage

In [26]:
df[df["convo_turn"].apply(lambda x: "user" in x.lower())].pivot_table(index="treatment", values='gratitude_flag')

,gratitude_flag
treatment,
Ingroup Agree,0.022989
Ingroup Disagree,0.014615
Outgroup Agree,0.032516
Outgroup Disagree,0.013385


In [27]:
df[df["convo_turn"].apply(lambda x: "llm" in x.lower())].pivot_table(index="treatment", values='gratitude_flag')

,gratitude_flag
treatment,
Ingroup Agree,0.104242
Ingroup Disagree,0.176437
Outgroup Agree,0.137793
Outgroup Disagree,0.169188


### Hedge usage

In [38]:
df[df["convo_turn"].apply(lambda x: "user" in x.lower())].pivot_table(index="treatment", values='hedge_flag')

,hedge_flag
treatment,
Ingroup Agree,0.740472
Ingroup Disagree,0.762788
Outgroup Agree,0.748431
Outgroup Disagree,0.799777


In [37]:
df[df["convo_turn"].apply(lambda x: "llm" in x.lower())].pivot_table(index="treatment", values='hedge_flag')

,hedge_flag
treatment,
Ingroup Agree,0.987879
Ingroup Disagree,0.993236
Outgroup Agree,0.989708
Outgroup Disagree,0.998880


### Vader sentiment

In [39]:
df[df["convo_turn"].apply(lambda x: "user" in x.lower())].pivot_table(index="treatment", values='positive_polarity')

,positive_polarity
treatment,
Ingroup Agree,0.631579
Ingroup Disagree,0.541315
Outgroup Agree,0.618939
Outgroup Disagree,0.542666


In [40]:
df[df["convo_turn"].apply(lambda x: "llm" in x.lower())].pivot_table(index="treatment", values='positive_polarity')

,positive_polarity
treatment,
Ingroup Agree,0.921212
Ingroup Disagree,0.866967
Outgroup Agree,0.974843
Outgroup Disagree,0.885154


In [18]:
eval(df['pos_tag_counts'].iloc[0])

NameError: name 'Counter' is not defined

In [ ]:
df['pos_tags'] = df['clean_body'].apply(lambda x: pos_tag(word_tokenize(x, language='english')))